In [ ]:
# Cell 1 — Install
# Runtime > Change runtime type > T4 GPU before running
!pip install timm albumentations scikit-learn pillow tqdm datasets requests -q
print("Done. Run Cell 2.")

In [ ]:
# Cell 2 — Download authentic real photos
# Sources: MIRFLICKR → COCO → Wikimedia → RAISE-1k → CIFAR-10 (fallback)
# Target: 600 train, 250 eval (70/30 routing), dedup by MD5, skip <300x300
import os
import io
import random
import hashlib
import time
from pathlib import Path
from PIL import Image
import requests

TRAIN_DIR = Path("/content/real_v5/train")
EVAL_DIR  = Path("/content/real_v5/eval")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TARGET = 600
EVAL_TARGET  = 250

seen_md5s   = set()
train_count = 0
eval_count  = 0

def _done():
    return train_count >= TRAIN_TARGET and eval_count >= EVAL_TARGET

def _route():
    global train_count, eval_count
    if train_count >= TRAIN_TARGET:
        return "eval"
    if eval_count >= EVAL_TARGET:
        return "train"
    total = train_count + eval_count
    if total == 0 or train_count / (total + 1e-9) < 0.7:
        return "train"
    return "eval"

def _save_pil(img, source):
    """Validate, dedup, save. Returns True if saved."""
    global train_count, eval_count
    if img.width < 300 or img.height < 300:
        return False
    img_rgb = img.convert("RGB")
    buf = io.BytesIO()
    img_rgb.save(buf, format="JPEG", quality=90)
    raw = buf.getvalue()
    md5 = hashlib.md5(raw).hexdigest()
    if md5 in seen_md5s:
        return False
    seen_md5s.add(md5)
    split   = _route()
    out_dir = TRAIN_DIR if split == "train" else EVAL_DIR
    idx     = train_count if split == "train" else eval_count
    tag     = "tr" if split == "train" else "ev"
    fname   = f"{source}_{tag}_{idx:06d}.jpg"
    out_dir.joinpath(fname).write_bytes(raw)
    if split == "train":
        train_count += 1
    else:
        eval_count += 1
    return True

# ── Source 1: MIRFLICKR ───────────────────────────────────────────────────────
print("Source 1 — MIRFLICKR")
if not _done():
    try:
        from datasets import load_dataset
        ds = load_dataset("nlphuji/flickr30k", split="test",
                          streaming=True)
        for ex in ds:
            if _done():
                break
            try:
                img = ex.get("image") or ex.get("img")
                if img is None:
                    continue
                if isinstance(img, dict):
                    img = Image.open(io.BytesIO(img.get("bytes", b"")))
                if not isinstance(img, Image.Image):
                    img = Image.fromarray(img)
                _save_pil(img, "mirflickr")
            except Exception:
                continue
    except Exception as e:
        print(f"  MIRFLICKR failed: {e}")
print(f"  After MIRFLICKR  train={train_count}  eval={eval_count}")

# ── Source 2: COCO 2017 ───────────────────────────────────────────────────────
print("Source 2 — COCO 2017")
if not _done():
    try:
        from datasets import load_dataset
        ds = load_dataset("detection-datasets/coco", split="train",
                          streaming=True)
        for ex in ds:
            if _done():
                break
            try:
                img = ex.get("image")
                if img is None:
                    continue
                if isinstance(img, dict):
                    img = Image.open(io.BytesIO(img.get("bytes", b"")))
                if not isinstance(img, Image.Image):
                    img = Image.fromarray(img)
                _save_pil(img, "coco")
            except Exception:
                continue
    except Exception as e:
        print(f"  COCO failed: {e}")
print(f"  After COCO       train={train_count}  eval={eval_count}")

# ── Source 3: Wikimedia Commons ───────────────────────────────────────────────
print("Source 3 — Wikimedia Commons")
if not _done():
    try:
        API     = "https://commons.wikimedia.org/w/api.php"
        HEADERS = {"User-Agent": "TruthLens/1.0"}
        resp = requests.get(API, params={
            "action": "query", "list": "recentchanges",
            "rcnamespace": 6, "rctype": "new",
            "rclimit": 500, "format": "json",
        }, headers=HEADERS, timeout=20)
        titles = [rc["title"]
                  for rc in resp.json().get("query", {}).get("recentchanges", [])]
        for title in titles:
            if _done():
                break
            try:
                info = requests.get(API, params={
                    "action": "query", "titles": title,
                    "prop": "imageinfo", "iiprop": "url",
                    "format": "json",
                }, headers=HEADERS, timeout=20).json()
                for page in info.get("query", {}).get("pages", {}).values():
                    url = page.get("imageinfo", [{}])[0].get("url", "")
                    if not url.lower().endswith((".jpg", ".jpeg", ".png")):
                        continue
                    r = requests.get(url, headers=HEADERS, timeout=30)
                    if r.status_code == 200:
                        _save_pil(Image.open(io.BytesIO(r.content)), "wikimedia")
                time.sleep(0.2)
            except Exception:
                continue
    except Exception as e:
        print(f"  Wikimedia failed: {e}")
print(f"  After Wikimedia  train={train_count}  eval={eval_count}")

# ── Source 4: RAISE-1k raw DSLR ───────────────────────────────────────────────
print("Source 4 — RAISE-1k")
if not _done():
    for n in range(1, 251):
        if _done():
            break
        url = f"http://loki.disi.unitn.it/RAISE/data/1k/{n:04d}.JPG"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                _save_pil(Image.open(io.BytesIO(r.content)), "raise")
        except Exception:
            pass
print(f"  After RAISE-1k   train={train_count}  eval={eval_count}")

# ── Source 5: CIFAR-10 (guaranteed fallback) ──────────────────────────────────
print("Source 5 — CIFAR-10 (fallback)")
if not _done():
    try:
        from datasets import load_dataset
        ds = load_dataset("uoft-cs/cifar10", split="train")
        for ex in ds:
            if _done():
                break
            try:
                img = ex.get("img") or ex.get("image")
                if img is None:
                    continue
                if isinstance(img, dict):
                    img = Image.open(io.BytesIO(img.get("bytes", b"")))
                if not isinstance(img, Image.Image):
                    img = Image.fromarray(img)
                img = img.resize((256, 256), Image.BILINEAR)
                _save_pil(img, "cifar10")
            except Exception:
                continue
    except Exception as e:
        print(f"  CIFAR-10 failed: {e}")
print(f"  After CIFAR-10   train={train_count}  eval={eval_count}")

print(f"\ntrain={train_count}  eval={eval_count}")
print("Run Cell 3.")

In [ ]:
# Cell 3 — Build data_v5 directory
# Upload data_v4.zip when prompted, then the rest runs automatically.
import zipfile, shutil
from pathlib import Path
from google.colab import files as colab_files

DATA_V4_DIR = Path("/content/data_v4")

if not DATA_V4_DIR.exists():
    print("Upload data_v4.zip when the dialog appears...")
    uploaded = colab_files.upload()          # pick D:\data_v4.zip
    zip_name = next(iter(uploaded))
    zip_path = Path("/content") / zip_name
    zip_path.write_bytes(uploaded[zip_name])
    print(f"Extracting {zip_name}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    zip_path.unlink()
    # Handle the case where zip contains a single top-level folder named data_v4
    extracted = [p for p in Path("/content").iterdir()
                 if p.is_dir() and p.name not in ("real_v5", "outputs_v5", "sample_data")]
    if not DATA_V4_DIR.exists() and len(extracted) == 1:
        extracted[0].rename(DATA_V4_DIR)
    print(f"data_v4 ready: {DATA_V4_DIR}")
else:
    print(f"data_v4 already at {DATA_V4_DIR}")

import hashlib
import shutil
from pathlib import Path

DATA_V4_DIR = Path("/content/data_v4")
DATA_V5_DIR = Path("/content/data_v5")

if not DATA_V4_DIR.exists():
    print(f"ERROR: {DATA_V4_DIR} does not exist.")
    print("Mount Google Drive or upload data_v4/ before running this cell.")
    raise SystemExit("Stopped — data_v4 missing.")

# Create v5 structure
for sub in ["train/ai", "train/real", "val/ai", "val/real"]:
    (DATA_V5_DIR / sub).mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def copy_dir(src: Path, dst: Path) -> int:
    count = 0
    if not src.exists():
        print(f"  WARNING: {src} not found — skipping")
        return 0
    for f in src.iterdir():
        if f.suffix.lower() in IMG_EXTS:
            shutil.copy2(f, dst / f.name)
            count += 1
    return count

# Step 3: Copy AI images unchanged
ai_train = copy_dir(DATA_V4_DIR / "train" / "ai", DATA_V5_DIR / "train" / "ai")
ai_val   = copy_dir(DATA_V4_DIR / "val"   / "ai", DATA_V5_DIR / "val"   / "ai")

# Step 4: Copy new real photos
real_train = copy_dir(Path("/content/real_v5/train"), DATA_V5_DIR / "train" / "real")
real_val   = copy_dir(Path("/content/real_v5/eval"),  DATA_V5_DIR / "val"   / "real")

print(f"train/ai   : {ai_train}")
print(f"train/real : {real_train}")
print(f"val/ai     : {ai_val}")
print(f"val/real   : {real_val}")
print(f"Total train: {ai_train + real_train}  |  Total val: {ai_val + real_val}")

# Step 6: MD5 cross-check — zero overlap between train and val
def md5_set(directory: Path) -> dict:
    hashes = {}
    for f in directory.rglob("*"):
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            h = hashlib.md5(f.read_bytes()).hexdigest()
            hashes[h] = str(f)
    return hashes

print("\nRunning MD5 cross-check...")
train_hashes = {}
for sub in ["train/ai", "train/real"]:
    train_hashes.update(md5_set(DATA_V5_DIR / sub))

val_hashes = {}
for sub in ["val/ai", "val/real"]:
    val_hashes.update(md5_set(DATA_V5_DIR / sub))

overlaps = set(train_hashes) & set(val_hashes)
if overlaps:
    for h in overlaps:
        print(f"  OVERLAP: {train_hashes[h]} <-> {val_hashes[h]}")
    raise RuntimeError("Overlap found — fix before training")
else:
    print("Zero overlap confirmed — data is clean")

print("")
print("Add 30-50 of your own phone photos to /content/data_v5/train/real/ now.")
print("Upload via Colab Files panel (left sidebar). Then run Cell 4.")

In [ ]:
# Cell 4 — Train v5
# Fixes v4's three regressions:
#   1. Loads FULL v3 (backbone + head), not backbone-only
#   2. Lower LR (2e-5 vs 1.2e-4) and milder regularization
#   3. Authentic camera photos as real class (from Cell 2/3)
import io
import random
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── T4 performance flags (required) ──────────────────────────────────────────
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# ── Config ───────────────────────────────────────────────────────────────────
CFG = {
    "data_dir":     "/content/data_v5",
    "output_dir":   "/content/outputs_v5",
    "prev_ckpt":    "/content/best_model_v3.pth",
    "model":        "efficientnet_b4",
    "img_size":     224,
    "batch_size":   32,
    "epochs":       20,       # was 8 — need more steps starting from ImageNet pretrained
    "lr":           5e-4,     # head LR during frozen phase (was 2e-5 — too slow starting from scratch)
    "weight_decay": 1e-4,
    "unfreeze_at":  4,        # was 2 — give head 4 epochs to learn before exposing backbone
    "num_workers":  2,
    "device":       "cuda",
    "amp":          True,
    "mixup_alpha":  0.05,
    "label_smooth": 0.03,
    "seed":         42,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = CFG["amp"] and device.type == "cuda"
print(f"Device: {device}  AMP: {use_amp}")

# ── Upload v3 checkpoint (skipped if already exists in session) ───────────────
if Path(CFG["prev_ckpt"]).exists():
    print(f"v3 checkpoint already at {CFG['prev_ckpt']} — skipping upload")
else:
    try:
        from google.colab import files as colab_files
        print("Upload best_model_v3.pth when prompted")
        uploaded = colab_files.upload()
        v3_bytes = next(iter(uploaded.values()))
        with open(CFG["prev_ckpt"], "wb") as fh:
            fh.write(v3_bytes)
        print(f"Saved to {CFG['prev_ckpt']} ({len(v3_bytes)/1e6:.1f} MB)")
    except ImportError:
        raise FileNotFoundError(
            f"v3 checkpoint not found at {CFG['prev_ckpt']}. Upload manually."
        )

# ── Augmentation (moderate — less aggressive than v4) ─────────────────────────
train_aug = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.15, 0.15, p=0.4),
    A.HueSaturationValue(10, 20, 10, p=0.3),
    A.ImageCompression(quality_range=(60, 95), p=0.5),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(CFG["img_size"], CFG["img_size"]),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

class AIDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.paths, self.labels = [], []
        root = Path(root)
        for label, cls in [(1, "ai"), (0, "real")]:
            folder = root / cls
            if not folder.exists():
                continue
            for p in folder.iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    self.paths.append(p)
                    self.labels.append(label)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img    = np.array(Image.open(self.paths[idx]).convert("RGB"))
        tensor = self.transform(image=img)["image"]
        return tensor, self.labels[idx]

# ── Mixup ─────────────────────────────────────────────────────────────────────
def mixup_batch(images, labels, alpha):
    if alpha <= 0:
        return images, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(images.size(0), device=images.device)
    return lam * images + (1 - lam) * images[idx], labels, labels[idx], lam

# ── Model (MUST be identical in Cell 4 and Cell 5) ───────────────────────────
class TruthLensModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b4", pretrained=True,
            num_classes=0, global_pool="avg")
        dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(dim, 512), nn.BatchNorm1d(512),
            nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128),
            nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, 2))
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  All unfrozen | Trainable: {n:,}")

    def forward(self, x):
        return self.head(self.backbone(x))

# ── Datasets & loaders ────────────────────────────────────────────────────────
train_ds = AIDataset(f"{CFG['data_dir']}/train", train_aug)
val_ds   = AIDataset(f"{CFG['data_dir']}/val",   val_aug)
print(f"Train: {len(train_ds)} images  |  Val: {len(val_ds)} images")

# Weighted sampler — class balance only, NO gold-tier 3x weighting
class_counts = np.bincount(train_ds.labels)
weights  = [1.0 / class_counts[l] for l in train_ds.labels]
sampler  = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], sampler=sampler,
    num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=True)

# ── Initialise model ──────────────────────────────────────────────────────────
model = TruthLensModel().to(device)

# THE CRITICAL FIX — load FULL v3 (both backbone AND head)
# If v3 used a different backbone architecture (different feature dim),
# fall back to ImageNet pretrained B4 rather than crashing.
try:
    _safe_global = np.core.multiarray.scalar
except AttributeError:
    _safe_global = np._core.multiarray.scalar  # numpy >= 2.0
torch.serialization.add_safe_globals([_safe_global])
ckpt = torch.load(CFG["prev_ckpt"], map_location="cpu", weights_only=True)

current_sd  = model.state_dict()
v3_sd       = ckpt["model_state"]
compatible  = {k: v for k, v in v3_sd.items()
               if k in current_sd and v.shape == current_sd[k].shape}
skipped     = [k for k in v3_sd if k not in compatible]

if compatible:
    model.load_state_dict(compatible, strict=False)
    print(f"Loaded {len(compatible)}/{len(v3_sd)} param tensors from v3 "
          f"(AUC={ckpt.get('val_auc', 0):.4f})")
    if skipped:
        print(f"  Skipped {len(skipped)} tensors with shape mismatch — "
              f"e.g. {skipped[0]} {v3_sd[skipped[0]].shape} vs {current_sd.get(skipped[0], 'missing')}")
else:
    print("WARNING: v3 checkpoint has no compatible weights (different backbone).")
    print("  Falling back to ImageNet pretrained EfficientNet-B4 (pretrained=True already loaded).")
    print(f"  v3 head.0.weight shape: {v3_sd.get('head.0.weight', '?')} ")
    print("  Training with good data + mild regularisation is the v5 fix.")

# ── Optimizer, scheduler, scaler ──────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["epochs"])
scaler    = torch.amp.GradScaler("cuda", enabled=use_amp)
loss_fn   = nn.CrossEntropyLoss(label_smoothing=CFG["label_smooth"])

# ── Training loop ─────────────────────────────────────────────────────────────
history  = {"loss": [], "val_auc": [], "ai_rate": []}
best_auc = 0.0
best_path = Path(CFG["output_dir"]) / "best_model_v5.pth"

for epoch in range(1, CFG["epochs"] + 1):

    # Unfreeze backbone after unfreeze_at frozen epochs
    if epoch == CFG["unfreeze_at"] + 1:
        model.unfreeze_all()
        # Discriminative LRs: backbone 50x lower than head (standard fine-tune recipe)
        optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": CFG["lr"] * 0.02},
            {"params": model.head.parameters(),     "lr": CFG["lr"] * 0.2},
        ], weight_decay=CFG["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG["epochs"] - epoch)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # ── Train phase ──────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    n_batches  = 0
    for imgs, labs in train_loader:
        imgs, labs = imgs.to(device), labs.to(device)
        imgs, labs_a, labs_b, lam = mixup_batch(imgs, labs, CFG["mixup_alpha"])
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(imgs)
            loss   = (lam       * loss_fn(logits, labs_a) +
                      (1 - lam) * loss_fn(logits, labs_b))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches  += 1
    scheduler.step()
    train_loss = total_loss / max(n_batches, 1)

    # ── Validation phase ─────────────────────────────────────────────────────
    model.eval()
    all_probs, all_labels = [], []
    ai_c = ai_t = real_c = real_t = 0
    with torch.no_grad():
        for imgs, labs in val_loader:
            imgs = imgs.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(imgs)
            probs   = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds   = (probs >= 0.5).astype(int)
            labs_np = labs.numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labs_np.tolist())
            ai_mask   = labs_np == 1
            real_mask = labs_np == 0
            ai_c   += int((preds[ai_mask] == 1).sum())
            ai_t   += int(ai_mask.sum())
            real_c += int((preds[real_mask] == 0).sum())
            real_t += int(real_mask.sum())

    val_auc   = roc_auc_score(all_labels, all_probs)
    val_acc   = (np.array(all_probs) >= 0.5).astype(int)
    val_acc   = float((val_acc == np.array(all_labels)).mean())
    ai_rate   = (ai_c   / ai_t   * 100) if ai_t   > 0 else 0.0
    real_rate = (real_c / real_t * 100) if real_t > 0 else 0.0

    history["loss"].append(train_loss)
    history["val_auc"].append(val_auc)
    history["ai_rate"].append(ai_rate)

    print(f"Epoch {epoch:02d}/{CFG['epochs']:02d}  "
          f"loss={train_loss:.4f}  acc={val_acc:.4f}  "
          f"auc={val_auc:.4f}  AI={ai_rate:.1f}%  Real={real_rate:.1f}%")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "model_state": model.state_dict(),
            "val_auc":     val_auc,
            "epoch":       epoch,
            "cfg":         CFG,
        }, best_path)
        print(f"  Saved best_model_v5.pth (AUC={val_auc:.4f})")

# ── Plot training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["loss"]);    axes[0].set_title("Train Loss")
axes[1].plot(history["val_auc"]); axes[1].set_title("Val AUC")
axes[2].plot(history["ai_rate"]); axes[2].set_title("AI Detection Rate (%)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/training_curves_v5.png", dpi=120)
plt.show()
print(f"\nBest AUC: {best_auc:.4f}")
print("Run Cell 5.")

In [ ]:
# Cell 5 — Calibrate + Evaluate + Deployment output
# TruthLensModel MUST be identical to Cell 4.
import json
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

CFG_V5 = {
    "output_dir": "/content/outputs_v5",
    "data_dir":   "/content/data_v5",
    "img_size":   224,
    "batch_size": 32,
    "num_workers": 2,
    "amp":         True,
}

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = CFG_V5["amp"] and device.type == "cuda"

# ── TruthLensModel (identical to Cell 4) ──────────────────────────────────────
class TruthLensModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b4", pretrained=True,
            num_classes=0, global_pool="avg")
        dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(dim, 512), nn.BatchNorm1d(512),
            nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128),
            nn.GELU(), nn.Dropout(0.25),
            nn.Linear(128, 2))
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))

# ── Step 1: Load best_model_v5.pth ────────────────────────────────────────────
try:
    _safe_global = np.core.multiarray.scalar
except AttributeError:
    _safe_global = np._core.multiarray.scalar
torch.serialization.add_safe_globals([_safe_global])

v5_path = Path(CFG_V5["output_dir"]) / "best_model_v5.pth"
ckpt    = torch.load(str(v5_path), map_location="cpu", weights_only=True)
model   = TruthLensModel().to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded v5 (val AUC from training: {ckpt.get('val_auc', 0):.4f})")

# ── Val dataset ───────────────────────────────────────────────────────────────
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

class AIDataset(Dataset):
    def __init__(self, root, transform):
        self.transform = transform
        self.paths, self.labels = [], []
        for label, cls in [(1, "ai"), (0, "real")]:
            folder = Path(root) / cls
            if not folder.exists():
                continue
            for p in folder.iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    self.paths.append(p)
                    self.labels.append(label)
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img    = np.array(Image.open(self.paths[idx]).convert("RGB"))
        tensor = self.transform(image=img)["image"]
        return tensor, self.labels[idx]

val_aug = A.Compose([
    A.Resize(CFG_V5["img_size"], CFG_V5["img_size"]),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])
val_ds     = AIDataset(f"{CFG_V5['data_dir']}/val", val_aug)
val_loader = DataLoader(val_ds, batch_size=CFG_V5["batch_size"],
                        shuffle=False, num_workers=CFG_V5["num_workers"],
                        pin_memory=True)
print(f"Val set: {len(val_ds)} images")

# ── Step 2: Collect raw logits ────────────────────────────────────────────────
all_logits, all_labels = [], []
with torch.no_grad():
    for imgs, labs in val_loader:
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(imgs)
        all_logits.append(logits.cpu())
        all_labels.extend(labs.tolist())

all_logits = torch.cat(all_logits, dim=0).float()
labels_np  = np.array(all_labels)

# ── Step 3: Fit TemperatureScaler with LBFGS 500 iterations ──────────────────
T_param       = nn.Parameter(torch.ones(1) * 1.5)
optimizer_cal = torch.optim.LBFGS([T_param], lr=0.01, max_iter=500)
logits_t      = all_logits.clone()
labels_t      = torch.tensor(labels_np, dtype=torch.long)

def cal_closure():
    optimizer_cal.zero_grad()
    loss = nn.CrossEntropyLoss()(logits_t / T_param, labels_t)
    loss.backward()
    return loss

optimizer_cal.step(cal_closure)
temperature = float(T_param.item())
print(f"Calibrated temperature: {temperature:.4f}")

# ── ECE ───────────────────────────────────────────────────────────────────────
def compute_ece(probs, labels, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i + 1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(labels[mask].mean() - probs[mask].mean())
    return float(ece / len(probs))

probs_raw = torch.softmax(all_logits, dim=1)[:, 1].detach().numpy()
probs_cal = torch.softmax(all_logits / T_param.detach(), dim=1)[:, 1].detach().numpy()
preds_cal = (probs_cal >= 0.5).astype(int)

# ── v5 metrics ────────────────────────────────────────────────────────────────
v5_auc      = roc_auc_score(labels_np, probs_cal)
v5_acc      = float((preds_cal == labels_np).mean())
ai_mask     = labels_np == 1
real_mask   = labels_np == 0
v5_ai_det   = float((preds_cal[ai_mask] == 1).mean() * 100)  if ai_mask.sum()   > 0 else 0.0
v5_real_acc = float((preds_cal[real_mask] == 0).mean() * 100) if real_mask.sum() > 0 else 0.0
v5_fp_rate  = float((preds_cal[real_mask] == 1).mean() * 100) if real_mask.sum() > 0 else 0.0
v5_ece_b    = compute_ece(probs_raw, labels_np)
v5_ece_a    = compute_ece(probs_cal, labels_np)

# ── Step 4: Print comparison table ────────────────────────────────────────────
V4 = {"auc": 0.8262, "acc": 0.7064, "ai": 47.2,
      "real": 93.6,  "fp": 6.4, "T": 1.2693,
      "ece_b": 0.1419, "ece_a": 0.1231}

print()
print("=" * 56)
print("  TRUTHLENS v5 — HONEST RESULTS")
print("=" * 56)
print(f"{'':22s}  {'v4 (broken)':>11s}  {'v5 (this run)':>13s}  {'change':>8s}")
print(f"{'AUC:':22s}  {V4['auc']:11.4f}  {v5_auc:13.4f}  {v5_auc - V4['auc']:+8.4f}")
print(f"{'Accuracy:':22s}  {V4['acc']:11.4f}  {v5_acc:13.4f}  {v5_acc - V4['acc']:+8.4f}")
print(f"{'AI detected:':22s}  {V4['ai']:10.1f}%  {v5_ai_det:12.1f}%  {v5_ai_det - V4['ai']:+7.1f}%")
print(f"{'Real correct:':22s}  {V4['real']:10.1f}%  {v5_real_acc:12.1f}%  {v5_real_acc - V4['real']:+7.1f}%")
print(f"{'False pos:':22s}  {V4['fp']:10.1f}%  {v5_fp_rate:12.1f}%  {v5_fp_rate - V4['fp']:+7.1f}%")
print(f"{'Temperature:':22s}  {V4['T']:11.4f}  {temperature:13.4f}")
print(f"{'ECE before:':22s}  {V4['ece_b']:11.4f}  {v5_ece_b:13.4f}  {v5_ece_b - V4['ece_b']:+8.4f}")
print(f"{'ECE after:':22s}  {V4['ece_a']:11.4f}  {v5_ece_a:13.4f}  {v5_ece_a - V4['ece_a']:+8.4f}")
print("=" * 56)

# ── Step 5: Save T.json and v5_eval_results.json ──────────────────────────────
out = Path(CFG_V5["output_dir"])

T_data = {
    "temperature": temperature,
    "ece_before":  round(v5_ece_b, 6),
    "ece_after":   round(v5_ece_a, 6),
    "val_auc":     round(v5_auc,   6),
    "model":       "best_model_v5.pth",
}
with open(out / "T.json", "w") as f:
    json.dump(T_data, f, indent=2)

eval_results = {
    "model":               "best_model_v5.pth",
    "temperature":         temperature,
    "val_images":          len(val_ds),
    "auc":                 round(v5_auc, 4),
    "accuracy":            round(v5_acc, 4),
    "ai_detection_rate":   round(v5_ai_det   / 100, 4),
    "real_accuracy":       round(v5_real_acc / 100, 4),
    "false_positive_rate": round(v5_fp_rate  / 100, 4),
    "ece_before":          round(v5_ece_b, 6),
    "ece_after":           round(v5_ece_a, 6),
}
with open(out / "v5_eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Saved T.json and v5_eval_results.json")

# ── Step 6: Deploy verdict ────────────────────────────────────────────────────
print()
if v5_auc > 0.90 and v5_ai_det > 75.0:
    print("VERDICT: DEPLOY v5 — significant improvement")
elif v5_auc > 0.87 and v5_ai_det > 65.0:
    print("VERDICT: MARGINAL IMPROVEMENT — v5 is better than v4 but real photos still limiting. Deploy anyway.")
else:
    print("VERDICT: DATA PROBLEM — authentic camera photos not sufficient. Contact for further diagnosis.")

# ── Step 7: Deployment commands ───────────────────────────────────────────────
print()
print("=" * 40)
print("TO DEPLOY v5")
print("=" * 40)
print("1. Download: best_model_v5.pth, T.json from /content/outputs_v5/")
print("2. Replace in repo backend folder:")
print("     backend/best_model_v5.pth")
print("     backend/T.json  (replaces v4's T.json — same filename)")
print("3. Update detector.py line:")
print("     _CHECKPOINT_PATH = _BASE / \"best_model_v5.pth\"")
print("4. git add backend/best_model_v5.pth backend/T.json backend/app/pipeline/detector.py")
print("5. git commit -m \"deploy v5: authentic real photos fix AI detection regression\"")
print("6. git push")
print("7. HuggingFace Space auto-redeploys on push to main")

# ── Download files ────────────────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    print("\nDownloading output files...")
    for fname in ["best_model_v5.pth", "T.json", "v5_eval_results.json", "training_curves_v5.png"]:
        fpath = out / fname
        if fpath.exists():
            colab_files.download(str(fpath))
            print(f"  Downloaded: {fname}")
        else:
            print(f"  Not found:  {fname}")
except ImportError:
    print(f"\nNot in Colab. Find output files at: {out}")